In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [6]:
class WarmupPolyLR:
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6, power=0.9):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.power = power
        self.current_epoch = 0
        
    def step(self):
        self.current_epoch += 1
        lr = self._get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr
    
    def _get_lr(self):
        if self.current_epoch < self.warmup_epochs:
            alpha = self.current_epoch / self.warmup_epochs
            return self.base_lr * alpha
        else:
            alpha = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            return (self.base_lr - self.min_lr) * (1 - alpha) ** self.power + self.min_lr
            
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']


class VehicleColorDataset(Dataset):
    def __init__(self, data_dir, transform=None, split='train', add_other_class=True):
        """
        Args:
            data_dir (string): Root directory for the dataset
            transform (callable, optional): Optional transform to be applied on a sample
            split (string): 'train' or 'val' split
            add_other_class (bool): Whether to add an "other" class dimension
        """
        self.data_dir = data_dir
        self.transform = transform
        self.split = split
        self.add_other_class = add_other_class
        
        self.colors = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        self.colors.sort()
        
        if add_other_class:
            self.colors.append("Other")
            
        self.color_to_idx = {color: idx for idx, color in enumerate(self.colors)}
        
        print(f"Found {len(self.colors) - (1 if add_other_class else 0)} color classes from data: {self.colors}")
        if add_other_class:
            print(f"Added extra 'Other' class dimension (index {len(self.colors)-1})")
        
        self.images = []
        self.labels = []
        
        for color in self.colors:
            if color == "Other":
                continue
                
            color_dir = os.path.join(data_dir, color)
            image_files = [f for f in os.listdir(color_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            for img_file in image_files:
                self.images.append(os.path.join(color_dir, img_file))
                self.labels.append(self.color_to_idx[color])
        
        if split == 'train' or split == 'val':
            indices = list(range(len(self.images)))
            train_indices, val_indices = train_test_split(
                indices, test_size=0.2, stratify=self.labels, random_state=42
            )
            
            if split == 'train':
                self.images = [self.images[i] for i in train_indices]
                self.labels = [self.labels[i] for i in train_indices]
            else:  # val
                self.images = [self.images[i] for i in val_indices]
                self.labels = [self.labels[i] for i in val_indices]
        
        print(f"{split} set: {len(self.images)} images")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            
            if self.transform:
                image = self.transform(image)
                
            return image, label
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            if self.transform:
                placeholder = Image.new('RGB', (224, 224), (0, 0, 0))
                return self.transform(placeholder), label
            return torch.zeros((3, 224, 224)), label


def prepare_data(data_dir, add_other_class=True):
    train_dataset = VehicleColorDataset(
        data_dir=data_dir,
        transform=data_transforms['train'],
        split='train',
        add_other_class=add_other_class
    )
    
    val_dataset = VehicleColorDataset(
        data_dir=data_dir,
        transform=data_transforms['val'],
        split='val',
        add_other_class=add_other_class
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4
    )
    
    return train_loader, val_loader, train_dataset.color_to_idx, train_dataset.colors

def initialize_model(num_classes):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    
    for param in list(model.parameters())[:-20]:
        param.requires_grad = False
    
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    return model

def train_model(model, train_loader, val_loader, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    scheduler = WarmupPolyLR(
        optimizer, 
        warmup_epochs=WARMUP_EPOCHS, 
        total_epochs=NUM_EPOCHS,
        base_lr=LEARNING_RATE
    )
    
    best_val_acc = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            current_lr = scheduler.get_lr()
            
            train_bar.set_postfix(loss=loss.item(), acc=correct/total, lr=current_lr)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        
        current_lr = scheduler.step()
        
        print(f'Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, LR: {current_lr:.6f}')
        
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_bar.set_postfix(loss=loss.item(), acc=val_correct/val_total)
        
        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_correct / val_total
        print(f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, './color_model/best_color_model.pth')
            print(f"Model saved with accuracy: {best_val_acc:.4f}")
            
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_current_epoch': scheduler.current_epoch,
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, f'./color_model/color_checkpoint_epoch_{epoch+1}.pth')
            print(f"Checkpoint saved at epoch {epoch+1}")

def evaluate_model(model, test_loader, color_names):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    pred_colors = [color_names[idx] for idx in all_preds]
    true_colors = [color_names[idx] for idx in all_labels]
    
    accuracy = sum(p == t for p, t in zip(all_preds, all_labels)) / len(all_preds)
    print(f"Test Accuracy: {accuracy:.4f}")
    
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=color_names[:-1])
    
    print("Classification Report:")
    print(report)
    
    return pred_colors, true_colors


In [7]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-4

data_dir = '/data/ibk5106/dataset/vehicle_datasets/vehicle_color/car_color_dataset/train'

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


add_other_class = True

train_loader, val_loader, color_to_idx, color_names = prepare_data(
    data_dir, 
    add_other_class=add_other_class
)

model = initialize_model(num_classes=len(color_to_idx))
model = model.to(device)

train_model(model, train_loader, val_loader, NUM_EPOCHS)


checkpoint = torch.load('./color_model/best_color_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1} with validation accuracy: {checkpoint['val_acc']:.4f}")

pred_colors, true_colors = evaluate_model(model, val_loader, color_names)

for i in range(min(10, len(pred_colors))):
    print(f"True: {true_colors[i]}, Predicted: {pred_colors[i]}")

Using device: cuda
Found 11 color classes from data: ['Black', 'Blue', 'Brown', 'Cyan', 'Green', 'Grey', 'Orange', 'Red', 'Violet', 'White', 'Yellow', 'Other']
Added extra 'Other' class dimension (index 11)
train set: 3041 images
Found 11 color classes from data: ['Black', 'Blue', 'Brown', 'Cyan', 'Green', 'Grey', 'Orange', 'Red', 'Violet', 'White', 'Yellow', 'Other']
Added extra 'Other' class dimension (index 11)
val set: 761 images


/tmp/ipykernel_3203074/1085709684.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./color_model/best_color_model.pth')


Loaded best model from epoch 94 with validation accuracy: 0.9330


Evaluating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:00<00:00, 38.59it/s]


Test Accuracy: 0.9330
Classification Report:
              precision    recall  f1-score   support

       Black       0.97      0.96      0.97        80
        Blue       0.87      0.93      0.90        57
       Brown       0.86      0.80      0.83        30
        Cyan       0.94      0.89      0.91        53
       Green       1.00      0.94      0.97        54
        Grey       0.90      0.99      0.94        67
      Orange       0.89      0.97      0.93       112
         Red       0.94      0.90      0.92       105
      Violet       0.94      0.88      0.91        33
       White       0.99      0.98      0.98        82
      Yellow       0.95      0.90      0.92        88

    accuracy                           0.93       761
   macro avg       0.93      0.92      0.93       761
weighted avg       0.93      0.93      0.93       761

True: Cyan, Predicted: Cyan
True: Black, Predicted: Grey
True: Yellow, Predicted: Yellow
True: Grey, Predicted: Grey
True: Orange, Predicted: 